# Sentinel Vision — Full Demo (Phase 4)

Full-stack video surveillance with all analytics wired in:
detection → tracking → zones → gates → dwell → loitering → abandoned objects
→ calibration → interactions → vehicle ANPR → scene understanding →
**face recognition → event correlation → trajectory prediction**

---
## Setup

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/basil-sami/sentinel-vision"
BRANCH = "phase-4-vehicle-intelligence"
REPO_DIR = "sentinel-vision"

if not Path(REPO_DIR).exists():
    !git clone --branch {BRANCH} --single-branch {REPO_URL}
else:
    %cd {REPO_DIR}
    !git checkout {BRANCH} && git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip install -q easyocr insightface

try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass
print("\nReady.")

## Video Source

In [ ]:
DEFAULT_VIDEO = "The CCTV People Demo 2.mp4"
DEFAULT_URL = "https://www.youtube.com/watch?v=GJNjaRJWVP8"

if not Path(DEFAULT_VIDEO).exists():
    print("Downloading test video...")
    !yt-dlp -f 'bestvideo[height<=1080]+bestaudio/best[height<=1080]' \
        --merge-output-format mp4 -o "{DEFAULT_VIDEO}" "{DEFAULT_URL}"

if Path(DEFAULT_VIDEO).exists() and Path(DEFAULT_VIDEO).stat().st_size > 0:
    input_video = DEFAULT_VIDEO
    print(f"Using: {input_video}")
else:
    print("Upload a CCTV video file.")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        input_video = list(uploaded.keys())[0]
        print(f"Using: {input_video}")
    else:
        raise FileNotFoundError("No video provided.")

## TensorRT Export (GPU only)

Exports YOLO11m to TensorRT FP16 engine (~3 min).
Switch `--model-size` to `xlarge` if you need maximum accuracy.

In [ ]:
import torch
MODEL_SIZE = "medium"  # Change to "xlarge" for max accuracy

if torch.cuda.is_available():
    from src.optimization.tensorrt_export import export_to_engine, has_engine
    if not has_engine("yolo11", MODEL_SIZE):
        print(f"Exporting YOLO11{MODEL_SIZE} to TensorRT...")
        path = export_to_engine(model_family="yolo11", model_size=MODEL_SIZE, device=0)
        print(f"Done: {path}")
    else:
        print(f"YOLO11{MODEL_SIZE} engine exists.")
else:
    print("CUDA not available — using PyTorch (CPU).")
    MODEL_SIZE = "nano"

## Face Gallery (Optional)

Register known faces so the pipeline recognizes them by name.
Upload photos and run this cell. Skip if you have no gallery.

In [ ]:
from src.analytics.face_recognition import FaceRecognizer
fr = FaceRecognizer(device="cuda" if torch.cuda.is_available() else "cpu")

if not fr.available:
    print("insightface not available — face recognition will be skipped at runtime.")
elif fr.gallery.known_names():
    print(f"Gallery has {len(fr.gallery.known_names())} known face(s):")
    for name in fr.gallery.known_names():
        print(f"  - {name}")
else:
    print("No known faces registered. Upload photos to add.")
    print("Example: fr.add_known_face_from_file('Alice', 'alice.jpg')")

## Run Full Pipeline

All analytics active: detection (YOLO11m TensorRT FP16), tracking (BotSORT + ReID),
zones, gates, dwell, loitering, abandoned objects, calibration, interactions,
evidence capture, vehicle ANPR (progressive enrichment), scene understanding
(carrying, overloaded), face recognition, trajectory prediction,
identity confidence tracking, event correlation, and time synchronization.

In [ ]:
import json
from src.pipeline import analyze_video

zone_config = {}
if Path("configs/demo_zones.json").exists():
    zone_config = json.load(open("configs/demo_zones.json"))

calibration_config = {}
if Path("configs/demo_calibration.json").exists():
    calibration_config = json.load(open("configs/demo_calibration.json"))

result = analyze_video(
    video_path=input_video,
    output_dir="outputs",
    model_family="yolo11",
    model_size=MODEL_SIZE,
    conf_threshold=0.4,
    device="cuda" if torch.cuda.is_available() else "cpu",
    max_frames=None,
    use_reid=True,
    reid_model="x1_0",
    track_thresh=0.4,
    match_thresh=0.7,
    track_low_thresh=0.1,
    track_buffer=450,
    zone_config=zone_config,
    calibration_config=calibration_config,
    capture_evidence=True,
    filter_stationary_objects=True,
    min_move_distance=20.0,
    use_tensorrt=torch.cuda.is_available(),
    plate_read_interval=10,
    use_cmc=False,
    reid_refresh_interval=50,
    reid_new_track_frames=3,
)
print("\nDone.")

## Full Report

Object counts, gate tallies, events, vehicle intelligence, scene understanding,
face identities, and correlated incidents.

In [ ]:
analytics = json.load(open("outputs/analytics.json"))

# Summary
summary = Path("outputs/summary.txt")
if summary.exists():
    print(summary.read_text())

# Vehicle intelligence
v = analytics.get("vehicles", {})
if v.get("total_vehicles", 0) > 0:
    print(f"\nVEHICLES: {v['total_vehicles']} total")
    print(f"  With plates:  {v['with_plates']}")
    print(f"  Without:      {v['without_plates']}")
    print(f"  Total visits: {v['total_visits']}")
    for vl in analytics.get("vehicle_list", [])[:5]:
        print(f"  {vl.get('plate') or 'unknown':>8s}  {vl['color']:>7s}  {vl['vehicle_type']:>10s}  "
              f"{vl['visit_count']} visit(s)")

# Scene understanding
se = analytics.get("scene_events", {})
if se.get("person_carrying", 0) > 0 or se.get("overloaded_vehicle", 0) > 0:
    print(f"\nSCENE EVENTS:")
    print(f"  Person carrying:  {se.get('person_carrying', 0)}")
    print(f"  Overloaded veh:   {se.get('overloaded_vehicle', 0)}")

# Face identities
idents = analytics.get("identities", [])
if idents:
    print(f"\nFACE RECOGNITIONS ({len(idents)}):")
    for id_ in idents:
        print(f"  Track {id_['track_id']}: {id_['name']} (conf={id_['confidence']})")

# Correlated incidents
incs = analytics.get("incidents", [])
if incs:
    print(f"\nCORRELATED INCIDENTS ({len(incs)}):")
    for inc in incs:
        print(f"  [{inc['severity'].upper()}] {inc['incident_type']}: {inc['summary']}")

# All events (last 10)
all_ev = analytics.get("events", [])
if all_ev:
    print(f"\nRecent events ({len(all_ev)} total, showing last 10):")
    for ev in all_ev[-10:]:
        print(f"  [{ev['type']}] {ev['message']}")

## Download Outputs

In [ ]:
from google.colab import files
files.download("outputs/output_tracking.mp4")
files.download("outputs/analytics.json")
files.download("outputs/summary.txt")

## Multi-Camera (4 feeds)

Generates 4 transformed copies and processes them in parallel.

In [ ]:
import subprocess
ORIGINAL = input_video
TEST_DIR = Path("test_videos")
TEST_DIR.mkdir(exist_ok=True)

FEEDS = [
    ("cam00_original", ORIGINAL),
    ("cam01_mirror_slow", TEST_DIR / "cam01_mirror_slow.mp4"),
    ("cam02_mirror_reverse", TEST_DIR / "cam02_mirror_reverse.mp4"),
    ("cam03_slow_reverse", TEST_DIR / "cam03_slow_reverse.mp4"),
]

if not FEEDS[1][1].exists():
    print("Generating transformed videos...")
    for name, path in FEEDS[1:]:
        filt = {}
        subprocess.run(["ffmpeg", "-y", "-i", ORIGINAL, "-vf",
                       {"cam01_mirror_slow": "hflip,setpts=2.0*PTS",
                        "cam02_mirror_reverse": "hflip,reverse",
                        "cam03_slow_reverse": "reverse,setpts=2.0*PTS"}[name],
                       "-an", "-c:v", "libx264", "-preset", "fast",
                       str(path)], capture_output=True)
        print(f"  {name}")

from src.optimization.multi_stream import MultiCameraPipeline

pipeline = MultiCameraPipeline(
    camera_configs=[
        {"video_path": str(vp), "name": nm,
         "model_size": MODEL_SIZE,
         "conf_threshold": 0.4,
         "device": "cuda" if torch.cuda.is_available() else "cpu",
         "use_tensorrt": torch.cuda.is_available(),
         "capture_evidence": True,
         "filter_stationary_objects": True,
         "min_move_distance": 20.0,
         "plate_read_interval": 10,
         "use_cmc": False,
         "reid_refresh_interval": 50,
         "reid_new_track_frames": 3,
        }
        for nm, vp in FEEDS
    ],
    output_dir="outputs/4feeds",
    max_workers=4,
)

report = pipeline.run()
print(f"\nAll feeds complete! Events: {report['event_count']}")

print(f"\n{'Camera':<20s} {'Frames':>8s} {'Tracks':>8s} {'Events':>8s} {'People':>8s}")
print("-" * 60)
for ck, cr in report.get("per_camera", {}).items():
    people = cr.get("object_counts", {}).get("person", 0)
    print(f"{ck:<20s} {cr.get('frames', 0):>8d} {cr.get('tracks', 0):>8d} {cr.get('events', 0):>8d} {people:>8d}")

mosaic_path = report.get("mosaic")
if mosaic_path and Path(mosaic_path).exists():
    print(f"\nMosaic: {mosaic_path} ({Path(mosaic_path).stat().st_size/1e6:.0f} MB)")